# Production-Grade Movie Recommender System
### From Unsupervised Learning → Recommendation Engineering → Deployment

This notebook is designed as the **next practical step after the clustering section** in the Machine Learning course.

We move from:

**Clustering → similarity → recommendation → evaluation → hybrid ranking → persistence → API-ready deployment**

The project uses the **MovieLens 100K** dataset because it is small enough for classroom laptops while still containing real user–movie interaction data.

> **Important concept:** recommender systems are not a single machine-learning algorithm. A production recommender is a system made of data pipelines, candidate generation, ranking, evaluation, business rules, monitoring and deployment.

### Learning objectives

By the end of this notebook, students should be able to:

1. Explain content-based, collaborative-filtering and hybrid recommendation.
2. Build an item-to-item similarity recommender.
3. Build a collaborative-filtering recommender from user–item interactions.
4. Create a hybrid recommender.
5. Handle cold-start users and items.
6. Evaluate recommendations with Precision@K, Recall@K, MAP@K and NDCG@K.
7. Avoid common recommendation-system data leakage.
8. Package a trained recommender into reusable artifacts.
9. Expose the recommender through an API-ready function.
10. Understand how the notebook becomes a production architecture.

## 0. Dataset choice

### Recommended classroom dataset: MovieLens 100K

The MovieLens 100K benchmark contains **100,000 ratings from 1,000 users on 1,700 movies**, making it a very good teaching dataset.

Kaggle:
- https://www.kaggle.com/datasets/trishna8/movielens-100k-dataset

### Other datasets students can use later

**TMDB 5000 Movie Dataset**  
Good for a content-based recommender using movie metadata, genres, keywords, cast, crew and descriptions. Kaggle has multiple versions and recommender-system notebooks built around it.

**Spotify song datasets**  
Good for music recommendation using audio features such as danceability, energy, tempo and valence. Kaggle currently has several Spotify datasets and recommendation projects.

### Suggested progression

| Level | Dataset | Main concept |
|---|---|---|
| Beginner | MovieLens 100K | Collaborative filtering |
| Intermediate | TMDB 5000 | Content-based recommendation |
| Advanced | Spotify | Hybrid/content + clustering |
| Production project | MovieLens 20M+ | Scalability and implicit feedback |

MovieLens 20M is also available through Kaggle and is useful once students understand the smaller dataset.

## 1. The production architecture we are building

A useful mental model is:

```text
                    ┌─────────────────────┐
                    │   Raw User Events   │
                    │ views / ratings /   │
                    │ clicks / purchases  │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Data Validation &   │
                    │ Feature Engineering │
                    └──────────┬──────────┘
                               │
              ┌────────────────┴────────────────┐
              ▼                                 ▼
   ┌────────────────────┐             ┌────────────────────┐
   │ Content Model      │             │ Collaborative      │
   │ TF-IDF / metadata  │             │ Filtering          │
   └─────────┬──────────┘             └─────────┬──────────┘
             │                                  │
             └────────────────┬─────────────────┘
                              ▼
                    ┌─────────────────────┐
                    │ Candidate Generation│
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │ Hybrid Ranker       │
                    │ + business rules    │
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │ Top-K Recommendations│
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │ API / Application   │
                    └─────────────────────┘
```

## 2. Environment and imports

For Kaggle, the dataset is normally mounted under `/kaggle/input/...`.

For local development, download the dataset and point `DATA_DIR` to the extracted folder.

We deliberately keep configuration in one place so the notebook teaches students how to move from exploratory code toward reproducible engineering.

In [ ]:
# --- Standard library imports ---
from pathlib import Path      # Path handles filesystem paths in an OS-independent way
import json                   # (unused directly here, but handy for saving configs/results)
import logging                # Used below to set up structured console logging
import warnings                # Lets us silence noisy library warnings

# --- Third-party imports ---
import joblib                 # Used later to save/load trained model artifacts to disk
import numpy as np            # Numerical arrays and math (e.g. log, argsort)
import pandas as pd           # DataFrames for loading and manipulating tabular data
import matplotlib.pyplot as plt  # Plotting library, used later for the evaluation bar chart

from scipy.sparse import csr_matrix                       # Efficient sparse matrix format for the user-item matrix
from sklearn.feature_extraction.text import TfidfVectorizer  # Converts genre text into TF-IDF vectors for content-based similarity
from sklearn.metrics.pairwise import cosine_similarity       # Computes cosine similarity between vectors/matrices
from sklearn.neighbors import NearestNeighbors                # k-NN model used for collaborative-filtering item similarity

# Suppress warning messages (e.g. from sklearn/pandas) so classroom output stays clean
warnings.filterwarnings("ignore")

# --- Global configuration constants ---
RANDOM_STATE = 42        # Fixed seed so sampling/splits are reproducible
TOP_K = 10                # Default number of recommendations to generate/evaluate
MIN_USER_RATINGS = 5      # Minimum number of ratings a user must have to be included (avoids extreme cold-start users)

# Change this path if you are running outside Kaggle.
DATA_DIR = Path("/kaggle/input/movielens-100k-dataset")  # Default location when running on Kaggle

# Optional local fallback: if the Kaggle path doesn't exist, look for a local "ml-100k" folder instead
if not DATA_DIR.exists():
    DATA_DIR = Path("./ml-100k")

# Directory where trained model artifacts (vectorizers, matrices, etc.) will be saved
ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)   # Create the folder if it doesn't already exist; do nothing if it does

# Configure logging so we get timestamped, leveled log messages instead of plain prints
logging.basicConfig(
    level=logging.INFO,                                   # Show INFO level and above
    format="%(asctime)s | %(levelname)s | %(message)s"    # Timestamp | level | message format
)
logger = logging.getLogger("recommender")   # Named logger for this notebook/module
logger.info("Data directory: %s", DATA_DIR.resolve())  # Log the resolved (absolute) data directory path

## 3. Load and inspect the data

MovieLens 100K provides:

- users
- movies
- ratings
- timestamps

The critical table for collaborative filtering is the **user–item interaction table**:

```text
user_id | movie_id | rating | timestamp
```

A recommendation system learns from interactions rather than a conventional `X` and `y` table.

In [ ]:
# Candidate folders to search for the MovieLens data files
possible_dirs = [DATA_DIR, DATA_DIR / "ml-100k"]
# Pick the first candidate folder that actually contains "u.data"; None if none match
data_dir = next((p for p in possible_dirs if (p / "u.data").exists()), None)

# Fail fast with a clear error if the dataset can't be found, rather than crashing later with a confusing error
if data_dir is None:
    raise FileNotFoundError(
        "Could not find u.data. Download/extract MovieLens 100K "
        "and update DATA_DIR."
    )

# Load the raw ratings interaction table (tab-separated, no header row in the file)
ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\\t",                                      # MovieLens 100K uses tabs as the delimiter
    names=["user_id", "movie_id", "rating", "timestamp"]  # Manually assign column names since file has none
)

# Column names for the movie metadata file (u.item), including 19 one-hot genre flags
movie_cols = [
    "movie_id", "title", "release_date", "video_release_date",
    "imdb_url", "unknown", "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime", "Documentary", "Drama",
    "Fantasy", "Film_Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci_Fi", "Thriller", "War", "Western"
]

# Load movie metadata (pipe-separated); latin-1 encoding is required because titles contain special characters
movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    names=movie_cols,
    encoding="latin-1"
)

# Quick visual sanity check of both tables
display(ratings.head())
display(movies.head())

In [ ]:
# Basic shape/size checks
print("Ratings shape:", ratings.shape)                              # (rows, columns) of the ratings table
print("Movies shape:", movies.shape)                                # (rows, columns) of the movies table
print("Unique users:", ratings["user_id"].nunique())                # How many distinct users rated something
print("Unique movies:", ratings["movie_id"].nunique())              # How many distinct movies were rated
print("Missing ratings:", ratings.isna().sum().sum())                # Total count of missing/NaN values across the ratings table
print("Rating range:", ratings["rating"].min(), "to", ratings["rating"].max())  # Sanity-check the min/max rating values

## 4. Data quality checks

Production systems should validate assumptions before training.

We check:

- duplicate interactions
- missing identifiers
- invalid ratings
- orphan movie IDs
- user and movie activity

In [ ]:
# --- Data integrity assertions: fail loudly if the raw data violates basic assumptions ---
assert ratings["user_id"].notna().all()          # No missing user IDs
assert ratings["movie_id"].notna().all()         # No missing movie IDs
assert ratings["rating"].between(1, 5).all()     # All ratings fall within the expected 1-5 scale

# Count rows that are exact duplicates on (user, movie, timestamp) - could indicate a logging bug upstream
duplicate_count = ratings.duplicated(
    subset=["user_id", "movie_id", "timestamp"]
).sum()

# Flag ratings that reference a movie_id not present in the movies table (orphaned/broken foreign keys)
orphan_movies = ~ratings["movie_id"].isin(movies["movie_id"])

print("Duplicate interaction rows:", duplicate_count)
print("Ratings referring to unknown movies:", orphan_movies.sum())

# Distribution of how many ratings each user has given (helps spot power users / sparse users)
print("\nRatings per user:")
display(ratings.groupby("user_id").size().describe())

# Distribution of how many ratings each movie has received (helps spot popular vs. long-tail movies)
print("\nRatings per movie:")
display(ratings.groupby("movie_id").size().describe())

# Part A — Build the evaluation strategy first

A major production mistake is training a recommender and then evaluating it on the same interactions it already saw.

For this classroom project, we create a **per-user holdout**:

- users with at least `MIN_USER_RATINGS` interactions are eligible
- one interaction is held out for testing
- the remaining interactions become training data

This teaches the idea of a **user-level interaction split** and avoids a simple random row split.

In [ ]:
# Keep only users who meet the minimum activity threshold (avoids extreme cold-start noise in training/eval)
eligible_users = (
    ratings.groupby("user_id")
    .size()                                    # Count ratings per user
    .loc[lambda s: s >= MIN_USER_RATINGS]        # Keep users with enough ratings
    .index                                      # Extract just the user_id values
)

# Filter the full ratings table down to only eligible users, then sort chronologically per user
eligible = ratings[ratings["user_id"].isin(eligible_users)].copy()
eligible = eligible.sort_values(["user_id", "timestamp"])

# Leave-one-out split: for each user, hold out their single most recent rating as the test set,
# and use everything else as training data. This avoids leaking future interactions into training.
test = eligible.groupby("user_id").tail(1).copy()
train = eligible.drop(test.index).copy()

print("Train interactions:", len(train))
print("Test interactions:", len(test))
print("Evaluation users:", test["user_id"].nunique())

## 5. Why the split matters

Suppose a user rated:

```text
Toy Story       5
Aladdin         5
Lion King       5
Jumanji         4
```

If `Jumanji` is hidden from the model, the recommendation system should attempt to retrieve it.

That creates the evaluation question:

> **Can the system recommend something the user actually interacted with later?**

This is much closer to the real product problem than predicting a rating for an already-known interaction.

# Part B — Baseline recommender

Before machine learning, build a simple baseline.

A production team should always know whether its sophisticated model beats a simple approach.

Our baseline recommends popular movies based on:

- number of ratings
- average rating
- a minimum number of ratings

We use a Bayesian-style weighted score so a movie with one perfect rating does not automatically beat a movie with hundreds of strong ratings.

In [ ]:
# Aggregate per-movie rating count and mean rating (from the TRAIN set only, to avoid leakage)
movie_stats = (
    train.groupby("movie_id")["rating"]
    .agg(["count", "mean"])
    .reset_index()
)

# C = overall average rating across all training interactions (the "prior" belief before seeing a movie's ratings)
C = train["rating"].mean()
# m = the 80th percentile of rating counts; movies need roughly this many ratings before their own average is trusted
m = movie_stats["count"].quantile(0.80)

# Bayesian/weighted average ("IMDB-style" formula): blends a movie's own mean rating with the global mean,
# weighted by how many ratings it has. Movies with few ratings get pulled toward the global average C,
# preventing a movie with one 5-star rating from outranking a movie with 500 ratings averaging 4.5.
movie_stats["weighted_score"] = (
    (movie_stats["count"] / (movie_stats["count"] + m)) * movie_stats["mean"]
    + (m / (movie_stats["count"] + m)) * C
)

# Rank movies by weighted score and attach titles - this becomes the popularity-based baseline recommender
popular_movies = (
    movie_stats.sort_values("weighted_score", ascending=False)
    .merge(movies[["movie_id", "title"]], on="movie_id")
)

display(popular_movies.head(10)[
    ["movie_id", "title", "count", "mean", "weighted_score"]
])

# Part C — Content-based recommendation

Content-based recommendation asks:

> **Which movies are similar to this movie?**

For MovieLens we can use genres as a simple content representation.

The pipeline is:

```text
Movie genres
    ↓
Text representation
    ↓
TF-IDF
    ↓
Cosine similarity
    ↓
Similar movies
```

This is directly connected to the concepts students already learned in preprocessing and unsupervised learning.

In [ ]:
# The 19 genre flag columns from the movies table (each is 0/1)
genre_cols = [
    "unknown", "Action", "Adventure", "Animation", "Children",
    "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
    "Film_Noir", "Horror", "Musical", "Mystery", "Romance",
    "Sci_Fi", "Thriller", "War", "Western"
]

content_movies = movies.copy()   # Work on a copy so we don't mutate the original movies DataFrame

# For a single movie row, collect the names of all genres flagged as 1, joined into a space-separated string
# e.g. a row with Action=1, Comedy=1 becomes "Action Comedy" - this text is what TF-IDF will vectorize
def make_genre_text(row):
    genres = [g for g in genre_cols if row[g] == 1]
    return " ".join(genres)

# Apply the function row-by-row to build a "genres_text" column for every movie
content_movies["genres_text"] = content_movies.apply(make_genre_text, axis=1)

# TF-IDF vectorizer turns each movie's genre text into a numeric vector.
# ngram_range=(1,2) captures both single genres and genre pairs (e.g. "Action Adventure") as features.
# min_df=2 ignores terms that appear in fewer than 2 movies, filtering out extremely rare noise.
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2
)

# Fit the vectorizer on all movies' genre text and transform it into a sparse TF-IDF matrix (movies x terms)
movie_tfidf = tfidf.fit_transform(content_movies["genres_text"])

print("TF-IDF matrix:", movie_tfidf.shape)

In [ ]:
# Precompute pairwise cosine similarity between every movie's TF-IDF vector (movies x movies matrix)
content_similarity = cosine_similarity(movie_tfidf, movie_tfidf)

# Lookup table mapping a movie_id to its row/column position inside content_similarity / movie_tfidf
movie_id_to_index = pd.Series(
    content_movies.index,
    index=content_movies["movie_id"]
).to_dict()

# Content-based recommender: given a movie, return the k most similar movies by genre-TF-IDF cosine similarity
def content_recommendations(movie_id, k=10):
    if movie_id not in movie_id_to_index:
        return pd.DataFrame(columns=["movie_id", "title", "similarity"])  # Unknown movie -> empty result

    idx = movie_id_to_index[movie_id]        # Row index of this movie in the similarity matrix
    scores = content_similarity[idx]          # Similarity of this movie to every other movie

    candidate_idx = np.argsort(scores)[::-1]              # Sort indices by similarity, descending
    candidate_idx = [i for i in candidate_idx if i != idx][:k]  # Drop the movie itself, keep top-k

    result = content_movies.iloc[candidate_idx][["movie_id", "title"]].copy()
    result["similarity"] = scores[candidate_idx]
    return result.reset_index(drop=True)

# Find the "Toy Story" row to use as a demo/example movie for testing the recommender
toy_story = movies[
    movies["title"].str.contains("Toy Story", case=False, na=False)
].iloc[0]

# Show the top 10 movies most similar to Toy Story by genre content
display(content_recommendations(int(toy_story["movie_id"]), k=10))

# Part D — Collaborative filtering

Content similarity knows about **movies**, but it does not know much about **individual users**.

Collaborative filtering uses the interaction matrix:

```text
             Movie A  Movie B  Movie C  Movie D
User 1          5        4        0        0
User 2          5        0        4        0
User 3          0        5        4        5
```

The key idea:

> Users who behaved similarly in the past may have similar preferences in the future.

For a practical classroom implementation we will use **item-item collaborative filtering**.

Instead of asking:

> Which users are similar to this user?

we ask:

> Which movies have similar interaction patterns to this movie?

In [ ]:
# Build a dense user-item ratings matrix: rows = users, columns = movies, values = rating (0 if unrated)
user_item = train.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating",
    fill_value=0
)

print("User-item matrix:", user_item.shape)

# Transpose so rows become movies (item-based collaborative filtering compares movies to each other),
# and convert to a sparse CSR matrix for memory/speed efficiency
item_user_matrix = csr_matrix(user_item.T.values)

# k-Nearest Neighbors model using cosine distance to find movies with similar rating patterns across users
knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",                                   # Brute-force search (fine at this dataset size)
    n_neighbors=min(50, item_user_matrix.shape[0])        # Cap neighbors at the number of available movies
)

knn.fit(item_user_matrix)   # Fit the k-NN index on the item (movie) vectors

# Keep the ordered list of movie IDs corresponding to each row of item_user_matrix
movie_id_list = user_item.columns.to_numpy()
# Reverse lookup: movie_id -> row index in item_user_matrix / movie_id_list
movie_id_to_row = {
    movie_id: idx for idx, movie_id in enumerate(movie_id_list)
}

In [ ]:
# Collaborative-filtering item similarity: given a movie, find the k movies most similar to it
# based on how users rated them (rather than genre metadata)
def collaborative_similar_items(movie_id, k=10):
    if movie_id not in movie_id_to_row:
        return pd.DataFrame(columns=["movie_id", "title", "similarity"])  # Unknown movie -> empty result

    row = movie_id_to_row[movie_id]                    # Row index of this movie in item_user_matrix
    n_neighbors = min(k + 1, len(movie_id_list))        # +1 because the movie itself will be its own nearest neighbor

    # Query the fitted k-NN model for the nearest movies (by cosine distance) to this movie's rating vector
    distances, indices = knn.kneighbors(
        item_user_matrix[row],
        n_neighbors=n_neighbors
    )

    ids = movie_id_list[indices[0]]        # Map neighbor row indices back to movie IDs
    similarities = 1 - distances[0]        # Convert cosine distance to cosine similarity (1 - distance)

    result = pd.DataFrame({
        "movie_id": ids,
        "similarity": similarities
    })

    # Remove the movie itself from its own neighbor list, then keep only the top k
    result = result[result["movie_id"] != movie_id].head(k)
    # Attach human-readable titles
    result = result.merge(
        movies[["movie_id", "title"]],
        on="movie_id",
        how="left"
    )

    return result[["movie_id", "title", "similarity"]].reset_index(drop=True)

# Demo: movies most similar to Toy Story according to collaborative filtering (user rating patterns)
display(collaborative_similar_items(int(toy_story["movie_id"]), k=10))

# Part E — Generate personalized candidates

Now we move from:

**movie → similar movies**

to:

**user → recommended movies**

For each movie a user liked, we retrieve similar movies.

Then we aggregate those signals.

This is an example of **candidate generation**.

In [ ]:
# Return everything a given user has rated in the training set, best-rated first
def get_user_history(user_id):
    return (
        train[train["user_id"] == user_id]
        .sort_values("rating", ascending=False)
    )

# Generate a pool of candidate movies for a user using item-based collaborative filtering:
# for each movie the user liked, pull in similar movies and accumulate a weighted score for each candidate
def generate_collaborative_candidates(user_id, candidate_pool=100):
    history = get_user_history(user_id)

    if history.empty:
        return pd.DataFrame(columns=["movie_id", "score"])   # No history -> no collaborative candidates

    scores = {}                                    # Accumulates score per candidate movie_id
    seen_movies = set(history["movie_id"])          # Movies the user already rated (exclude from candidates)

    liked = history[history["rating"] >= 4.0].head(30)   # Consider up to 30 of the user's highest-rated movies

    for _, row in liked.iterrows():
        movie_id = row["movie_id"]
        user_rating = row["rating"]

        # Find movies similar to this liked movie (via collaborative filtering)
        similar = collaborative_similar_items(movie_id, k=30)

        for _, item in similar.iterrows():
            candidate = int(item["movie_id"])

            if candidate in seen_movies:
                continue   # Don't recommend something the user has already rated

            # Weight the similarity by how strongly the user liked the source movie (normalized 0-1)
            contribution = float(item["similarity"]) * float(user_rating / 5.0)
            scores[candidate] = scores.get(candidate, 0) + contribution   # Accumulate across multiple liked movies

    if not scores:
        return pd.DataFrame(columns=["movie_id", "score"])

    # Convert the accumulated scores dict into a sorted DataFrame, capped at candidate_pool rows
    result = pd.DataFrame(
        [{"movie_id": movie_id, "score": score}
         for movie_id, score in scores.items()]
    )

    return result.sort_values("score", ascending=False).head(candidate_pool)

# Pick an example user (the first one in the test set) to demo the candidate generator
example_user = int(test["user_id"].iloc[0])

display(
    generate_collaborative_candidates(example_user, 20)
    .merge(movies[["movie_id", "title"]], on="movie_id", how="left")
    .head(10)
)

# Part F — Hybrid recommender

Real recommendation systems often combine multiple signals.

Our teaching model combines:

1. **Collaborative score** — users with similar interaction patterns
2. **Content score** — similarity of movie attributes
3. **Popularity score** — protects against obscure/noisy recommendations

The important engineering lesson is that recommendation is often a **ranking problem**.

We first create candidates, then calculate multiple features, then rank them.

In [ ]:
# Lookup table: movie_id -> its popularity (weighted_score) from the earlier popularity baseline
popularity_lookup = movie_stats.set_index("movie_id")["weighted_score"].to_dict()

# Min-max normalize a pandas Series to the 0-1 range so different score types can be combined fairly
def normalize_series(s):
    if len(s) == 0:
        return s
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)), index=s.index)   # Avoid divide-by-zero when all values are equal
    return (s - s.min()) / (s.max() - s.min())

# Hybrid recommender: blends collaborative filtering, content similarity, and popularity into one ranked list
def hybrid_recommend(user_id, k=10):
    history = get_user_history(user_id)

    # Cold-start fallback: brand-new user with no history gets the popularity baseline instead
    if history.empty:
        return popular_movies.head(k)[
            ["movie_id", "title", "weighted_score"]
        ].rename(columns={"weighted_score": "score"})

    # Get a larger pool of collaborative-filtering candidates than we need, so ranking has room to work with
    candidates = generate_collaborative_candidates(
        user_id,
        candidate_pool=max(100, k * 10)
    )

    # If collaborative filtering produced nothing useful, fall back to popularity again
    if candidates.empty:
        return popular_movies.head(k)[
            ["movie_id", "title", "weighted_score"]
        ].rename(columns={"weighted_score": "score"})

    # Normalize the raw collaborative score to 0-1 so it can be combined with other signals
    candidates["collab_score"] = normalize_series(candidates["score"])
    candidates["content_score"] = 0.0   # Placeholder, filled in below

    # Attach a popularity score to each candidate (default to global mean C if the movie has no stats)
    candidates["popularity_score"] = (
        candidates["movie_id"].map(popularity_lookup).fillna(C)
    )
    candidates["popularity_score"] = normalize_series(
        candidates["popularity_score"]
    )

    # The user's top 20 highly-rated movies, used as a "taste profile" for content similarity
    liked_movies = history[
        history["rating"] >= 4.0
    ].head(20)["movie_id"].tolist()

    # For each candidate, compute its best (max) content-based similarity to any of the user's liked movies
    for candidate_id in candidates["movie_id"]:
        sims = []

        for liked_id in liked_movies:
            if liked_id in movie_id_to_index and candidate_id in movie_id_to_index:
                a = movie_id_to_index[liked_id]
                b = movie_id_to_index[candidate_id]
                sims.append(content_similarity[a, b])

        if sims:
            candidates.loc[
                candidates["movie_id"] == candidate_id,
                "content_score"
            ] = max(sims)   # Use the single strongest content match rather than an average

    # Final hybrid score: weighted blend of collaborative (60%), content (30%), and popularity (10%) signals
    candidates["final_score"] = (
        0.60 * candidates["collab_score"]
        + 0.30 * candidates["content_score"]
        + 0.10 * candidates["popularity_score"]
    )

    # Sort by final score, keep the top k, and attach movie titles for display
    result = (
        candidates
        .sort_values("final_score", ascending=False)
        .head(k)
        .merge(movies[["movie_id", "title"]], on="movie_id", how="left")
    )

    return result[
        ["movie_id", "title", "collab_score",
         "content_score", "popularity_score", "final_score"]
    ].reset_index(drop=True)

# Demo: hybrid recommendations for the example user
display(hybrid_recommend(example_user, k=10))

# Part G — Recommendation evaluation

Classification accuracy is not the right metric for a Top-K recommender.

We care about questions such as:

- Did we retrieve the item the user actually liked?
- How many relevant items were retrieved?
- Were relevant items near the top?
- How much better is our model than a popularity baseline?

We implement:

- Precision@K
- Recall@K
- Average Precision@K
- NDCG@K

In [ ]:
# Precision@K: of the top-k recommended items, what fraction were actually relevant (liked) by the user?
def precision_at_k(recommended, relevant, k):
    recommended = list(recommended)[:k]
    relevant = set(relevant)

    if not recommended:
        return 0.0

    hits = sum(item in relevant for item in recommended)   # Count how many recommended items were relevant
    return hits / len(recommended)

# Recall@K: of all the relevant items that exist, what fraction did we manage to surface in the top-k?
def recall_at_k(recommended, relevant, k):
    recommended = list(recommended)[:k]
    relevant = set(relevant)

    if not relevant:
        return 0.0

    hits = sum(item in relevant for item in recommended)
    return hits / len(relevant)

# Average Precision@K: rewards placing relevant items EARLIER in the ranked list, not just including them
def average_precision_at_k(recommended, relevant, k):
    recommended = list(recommended)[:k]
    relevant = set(relevant)

    if not relevant:
        return 0.0

    hits = 0
    score = 0.0

    for rank, item in enumerate(recommended, start=1):
        if item in relevant:
            hits += 1
            score += hits / rank   # Precision at this rank position, added only when a hit occurs

    return score / min(len(relevant), k)   # Normalize by the best possible number of hits

# NDCG@K (Normalized Discounted Cumulative Gain): rewards relevant items near the top of the list,
# with a logarithmic discount for items appearing further down
def ndcg_at_k(recommended, relevant, k):
    recommended = list(recommended)[:k]
    relevant = set(relevant)

    dcg = 0.0
    for rank, item in enumerate(recommended, start=1):
        if item in relevant:
            dcg += 1 / np.log2(rank + 1)   # Discounted gain: hits count less the further down the list they are

    ideal_hits = min(len(relevant), k)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )   # IDCG = the best possible DCG if all relevant items were ranked first

    return dcg / idcg if idcg > 0 else 0.0   # Normalize actual DCG by the ideal DCG

## 7. Evaluate the popularity baseline

We start with the simplest possible recommendation engine.

If our ML system cannot beat this baseline, we should not deploy the ML system.

In [ ]:
# The fixed "always recommend the top-10 most popular movies" baseline, used for comparison
popular_ids = popular_movies["movie_id"].head(TOP_K).tolist()
# For each test user, the list of movie_ids they interacted with in the held-out test set (the "ground truth")
test_by_user = test.groupby("user_id")["movie_id"].apply(list).to_dict()

baseline_metrics = {
    "precision@10": [],
    "recall@10": [],
    "map@10": [],
    "ndcg@10": []
}

# Evaluate the popularity baseline against every test user's held-out relevant movie(s)
for user_id, relevant in test_by_user.items():
    baseline_metrics["precision@10"].append(
        precision_at_k(popular_ids, relevant, TOP_K)
    )
    baseline_metrics["recall@10"].append(
        recall_at_k(popular_ids, relevant, TOP_K)
    )
    baseline_metrics["map@10"].append(
        average_precision_at_k(popular_ids, relevant, TOP_K)
    )
    baseline_metrics["ndcg@10"].append(
        ndcg_at_k(popular_ids, relevant, TOP_K)
    )

# Average each metric across all users to get a single overall score per metric
baseline_result = {
    metric: float(np.mean(values))
    for metric, values in baseline_metrics.items()
}

baseline_result

## 8. Evaluate the hybrid recommender

For classroom speed, we evaluate a sample of users.

In a production system this evaluation would normally run as a scheduled batch job and write metrics to an experiment-tracking/monitoring system.

In [ ]:
# Sample up to 300 test users (for speed) to evaluate the hybrid model on, using a fixed seed for reproducibility
EVAL_USERS = test["user_id"].drop_duplicates().sample(
    min(300, test["user_id"].nunique()),
    random_state=RANDOM_STATE
)

model_metrics = {
    "precision@10": [],
    "recall@10": [],
    "map@10": [],
    "ndcg@10": []
}

# For each sampled user, generate hybrid recommendations and score them against their held-out relevant movie
for user_id in EVAL_USERS:
    relevant = test_by_user[int(user_id)]

    recs = hybrid_recommend(int(user_id), k=TOP_K)
    recommended_ids = recs["movie_id"].tolist()

    model_metrics["precision@10"].append(
        precision_at_k(recommended_ids, relevant, TOP_K)
    )
    model_metrics["recall@10"].append(
        recall_at_k(recommended_ids, relevant, TOP_K)
    )
    model_metrics["map@10"].append(
        average_precision_at_k(recommended_ids, relevant, TOP_K)
    )
    model_metrics["ndcg@10"].append(
        ndcg_at_k(recommended_ids, relevant, TOP_K)
    )

# Average each metric across all evaluated users
model_result = {
    metric: float(np.mean(values))
    for metric, values in model_metrics.items()
}

# Put the popularity baseline and hybrid model results side by side for direct comparison
comparison = pd.DataFrame(
    [baseline_result, model_result],
    index=["Popularity Baseline", "Hybrid Recommender"]
)

display(comparison)

In [ ]:
# Bar chart comparing the popularity baseline vs. hybrid recommender across all four metrics
comparison.plot(kind="bar", figsize=(12, 6))
plt.title("Recommendation Quality Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)                  # Keep the model-name labels horizontal for readability
plt.grid(axis="y", alpha=0.25)          # Light horizontal gridlines to make bar heights easier to compare
plt.show()

# Part H — Cold-start handling

A production recommender must handle users it has never seen.

### New user

We do not have enough interaction history.

Possible strategy:

```text
New user
   ↓
Ask for favourite genres / movies
   ↓
Content-based recommendations
   ↓
Blend with popular items
```

### New movie

No interaction history exists.

Possible strategy:

```text
New movie
   ↓
Use metadata / genres / description
   ↓
Content similarity
   ↓
Recommend to relevant users
```

This is why a hybrid system is usually more practical than relying entirely on collaborative filtering.

In [ ]:
# Cold-start recommender for brand-new users with no rating history yet.
# Falls back to the popularity ranking, optionally boosted by any genres the user says they prefer.
def recommend_for_new_user(k=10, preferred_genres=None):
    result = popular_movies.copy()
    result["score"] = result["weighted_score"]   # Start from the popularity score

    if preferred_genres:
        preferred_genres = [g.lower() for g in preferred_genres]

        # Simple teaching implementation.
        # A production version would build a proper genre preference vector.
        # Returns 1 if the movie shares at least one genre with the user's stated preferences, else 0
        def genre_match(movie_id):
            row = movies.loc[movies["movie_id"] == movie_id, genre_cols]
            if row.empty:
                return 0
            active = [
                genre.lower()
                for genre in genre_cols
                if int(row.iloc[0][genre]) == 1
            ]
            return int(any(g in active for g in preferred_genres))

        result["preference_bonus"] = result["movie_id"].apply(genre_match)
        result["score"] += 0.25 * result["preference_bonus"]   # Small boost for genre-matching movies

    return result.sort_values("score", ascending=False).head(k)[
        ["movie_id", "title", "score"]
    ]

# Demo: cold-start recommendations with no genre preference supplied (pure popularity ranking)
display(recommend_for_new_user(k=10))

# Part I — Model and artifact persistence

A notebook is **not** a production application.

A production workflow separates:

```text
Training
   ↓
Save model artifacts
   ↓
Application loads artifacts
   ↓
Recommendation request
   ↓
Prediction
```

We save the objects required for inference.

In [ ]:
# Bundle every object needed to serve recommendations at inference time into one dictionary.
# This is the "model artifact" that would normally be loaded by an API server in production.
artifacts = {
    "tfidf_vectorizer": tfidf,              # Fitted TF-IDF vectorizer for genre text
    "movie_tfidf": movie_tfidf,             # TF-IDF matrix for all movies
    "content_movies": content_movies,       # Movie metadata + genre text
    "movie_id_to_index": movie_id_to_index,  # Lookup for content similarity matrix rows
    "popular_movies": popular_movies,       # Precomputed popularity ranking
    "movie_id_to_row": movie_id_to_row,     # Lookup for collaborative filtering matrix rows
    "movie_id_list": movie_id_list,         # Ordered movie IDs matching the item-user matrix rows
}

artifact_path = ARTIFACT_DIR / "movie_recommender_artifacts.joblib"
joblib.dump(artifacts, artifact_path)   # Serialize the whole dictionary to a single file on disk

print("Saved:", artifact_path.resolve())
print("Size:", round(artifact_path.stat().st_size / 1024**2, 2), "MB")

## 9. Load the recommender artifacts

The inference application should not retrain the model.

It loads the previously trained artifacts.

In [ ]:
# Reload the saved artifacts from disk, simulating what an API/inference process would do on startup
loaded_artifacts = joblib.load(artifact_path)

print("Loaded artifact keys:")
print(list(loaded_artifacts.keys()))   # Confirm all expected objects came back successfully

# Part J — Build an inference interface

This is the interface an API or frontend would call.

The important design principle is:

> **The application should not need to know how the model was trained.**

It should simply call:

```python
recommend(user_id=123, k=10)
```

In [ ]:
# Unified inference entry point - this is the function an API endpoint would call.
# Routes the request to the appropriate recommender depending on which arguments are provided.
def recommend(user_id=None, movie_id=None, k=10):
    """
    Production-style inference interface.

    user_id:
        Return personalized recommendations.

    movie_id:
        Return similar movies.

    If neither is supplied:
        Return popular movies.
    """

    if user_id is not None:
        # Known/existing user -> personalized hybrid recommendations
        if user_id in train["user_id"].values:
            return hybrid_recommend(int(user_id), k=k)
        # Unknown user_id (not seen in training) -> cold-start fallback
        return recommend_for_new_user(k=k)

    if movie_id is not None:
        # No user context, but a movie was given -> "more like this" content-based recommendations
        return content_recommendations(int(movie_id), k=k)

    # No user or movie given at all -> just return the overall most popular movies
    return popular_movies.head(k)[
        ["movie_id", "title", "weighted_score"]
    ].rename(columns={"weighted_score": "score"})

# --- Demonstrate all three modes of the unified recommend() function ---
print("Personalized:")
display(recommend(user_id=example_user, k=5))   # Existing user -> hybrid recommendations

print("Similar movies:")
display(recommend(movie_id=int(toy_story["movie_id"]), k=5))   # Movie given -> content-based similar movies

print("Cold-start:")
display(recommend(k=5))   # Neither given -> popularity fallback

# Part K — API-ready design

The next production step is to expose the inference function through **FastAPI**.

A typical service would have endpoints such as:

```text
GET /health
GET /recommendations/user/{user_id}
GET /recommendations/movie/{movie_id}
```

A minimal API layer could look like this:

```python
from fastapi import FastAPI

app = FastAPI(title="Movie Recommendation API")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/recommendations/user/{user_id}")
def user_recommendations(user_id: int, k: int = 10):
    result = recommend(user_id=user_id, k=k)
    return result.to_dict(orient="records")

@app.get("/recommendations/movie/{movie_id}")
def movie_recommendations(movie_id: int, k: int = 10):
    result = recommend(movie_id=movie_id, k=k)
    return result.to_dict(orient="records")
```

### Important production separation

Do **not** put model training inside an API request.

Instead:

```text
Training job
    ↓
Artifacts / model registry
    ↓
API container
    ↓
Fast inference
```

> **Implemented for real:** the full FastAPI service (app factory, lifespan loading, validation, error handling) is walked through in **Part P5** below.

# Part L — Production architecture students should understand

A real recommender can evolve into:

```text
                     DATA SOURCES
        ┌───────────────┼────────────────┐
        ▼               ▼                ▼
     Ratings         Clicks          Purchases
        │               │                │
        └───────────────┼────────────────┘
                        ▼
                 Data Warehouse
                        │
                        ▼
              Feature Engineering
                        │
          ┌─────────────┴─────────────┐
          ▼                           ▼
   Content Features          Interaction Features
          │                           │
          ▼                           ▼
   Content Model              Collaborative Model
          │                           │
          └─────────────┬─────────────┘
                        ▼
                Candidate Generation
                        │
                        ▼
                   Re-ranking
                        │
               ┌────────┴────────┐
               ▼                 ▼
        Business Rules       Diversity
               │                 │
               └────────┬────────┘
                        ▼
                 Top-K Results
                        │
                        ▼
                  FastAPI / App
                        │
                        ▼
                      User
                        │
                        ▼
                 New interactions
                        │
                        └──────► Monitoring / Retraining
```

# Part M — What makes this production-grade?

The notebook has introduced the core pieces, but production engineering requires more than model accuracy.

### 1. Data engineering

- scheduled ingestion
- schema validation
- duplicate detection
- missing-data handling
- event timestamps
- feature freshness

### 2. Model engineering

- reproducible training
- versioned artifacts
- experiment tracking
- offline evaluation
- model rollback

### 3. Recommendation engineering

- candidate generation
- ranking
- diversity
- popularity controls
- business constraints
- cold-start strategy

### 4. API engineering

- FastAPI
- request validation
- authentication
- rate limiting
- logging
- health checks

### 5. Infrastructure

A mature deployment can use:

```text
Docker
    ↓
FastAPI
    ↓
Cloud / Kubernetes
    ↓
Redis / Feature Store
    ↓
Database / Data Warehouse
```

### 6. Monitoring

Monitor both:

**System metrics**
- latency
- throughput
- errors
- CPU/memory

**ML metrics**
- Precision@K
- Recall@K
- NDCG@K
- recommendation coverage
- diversity
- popularity bias
- drift
- click-through rate
- conversion rate

# Part N — Student challenge

## Challenge 1 — Improve the content model

Replace the simple genre representation with:

- movie title
- genres
- release year

Then compare recommendation quality.

---

## Challenge 2 — Add diversity

A recommender should not return ten almost-identical movies.

Create a diversity penalty so that recommendations from the same genre are not overrepresented.

---

## Challenge 3 — Add a real user profile

Instead of only using highly-rated movies, calculate a user's genre preference vector.

Example:

```text
User 42

Drama       0.82
Comedy      0.71
Romance     0.64
Action      0.20
Horror      0.05
```

Use this profile to rank movies.

---

## Challenge 4 — Build a Spotify recommender

Use a Spotify dataset containing audio features such as:

- danceability
- energy
- loudness
- speechiness
- acousticness
- instrumentalness
- liveness
- valence
- tempo

Apply:

```text
Scaling
   ↓
K-Means
   ↓
Song clusters
   ↓
Nearest-neighbor similarity
   ↓
Top-K recommendations
```

---

## Challenge 5 — Build a hybrid Spotify system

Combine:

**content similarity + song popularity + user listening history**

and evaluate it with Precision@K and Recall@K.

# Part O — Final project specification

## Build: "Tech-Africa Movie Recommendation Platform"

Students should deliver:

### Notebook 1 — Exploration

- EDA
- data quality
- visualization
- user/movie distributions

### Notebook 2 — Model development

- popularity baseline
- content-based model
- collaborative filtering
- hybrid recommender
- evaluation
- model comparison

### Python package

```text
recommender/
│
├── data.py
├── preprocessing.py
├── content_model.py
├── collaborative_model.py
├── ranking.py
├── evaluation.py
├── inference.py
└── config.py
```

### API

```text
GET /health
GET /recommendations/user/{user_id}
GET /recommendations/movie/{movie_id}
```

### Deployment

Students can containerize the API using Docker.

### Documentation

README must explain:

1. Problem
2. Dataset
3. Architecture
4. Algorithms
5. Evaluation
6. API
7. How to run locally
8. Limitations
9. Future improvements

# Part P — The remainder of the system

Parts A–O proved every idea **inside one notebook**. A notebook is great for learning, but it cannot be tested,
versioned cleanly, deployed, or run by someone else with one command. This part is the *engineering half*:
the same ideas, moved into a package and wrapped in scripts.

| File | Job | Notebook idea it replaces |
|---|---|---|
| `recommender/` (package) | data, split, models, ranking, metrics, inference | Parts B–J (functions) |
| `evaluate.py` | offline experiment: compare models, tune weights, write reports | Part G (evaluation) |
| `train.py` | fit on **all** data and save one artifact | Part I (persistence) |
| `api.py` | HTTP inference layer, never trains | Part K (API-ready design) |
| `Dockerfile` | package the API so it runs anywhere | Part L (architecture) |
| `tests/` + `.github/workflows/ci.yml` | automatic checks on every push | Part M (production-grade) |

```text
movie-recommender/
├── recommender/            # the library
│   ├── config.py           # every constant in one place
│   ├── data.py             # download + validate MovieLens
│   ├── preprocessing.py    # Index, temporal split, sparse matrices
│   ├── content_model.py    # genre cosine similarity
│   ├── collaborative_model.py  # ItemKNN, SVD (PureSVD)
│   ├── ranking.py          # popularity, random, normalise + blend
│   ├── evaluation.py       # rank-based metrics
│   └── inference.py        # Recommender: the object the API serves
├── tests/                  # 42 pytest tests
├── notebooks/              # exploration + model development
├── reports/                # CSVs, chart, best_config.json
├── evaluate.py  train.py  api.py
├── Dockerfile  .dockerignore  .gitignore
├── requirements.txt  ruff.toml  pytest.ini
└── .github/workflows/ci.yml
```

### The data flow (memorise this)

```text
evaluate.py ──► reports/best_config.json ──► train.py ──► artifacts/recommender.joblib ──► api.py ──► Docker
 (experiment)      (the hand-off file)       (final fit)         (the artifact)           (serving)
```

`best_config.json` is the contract between experimenting and training: `evaluate.py` *decides* which rating signal
and hybrid weights are best; `train.py` *obeys*. Nothing is hard-coded twice.

### What changed from the notebook version

| Topic | Notebook (Parts B–J) | Package (this part) |
|---|---|---|
| Split | latest rating per user → test | latest → **test**, second-latest → **validation**, rest → train |
| Tuning | hybrid weights fixed by hand (0.6 / 0.3 / 0.1) | weights **grid-searched on validation**, test touched once |
| Ranking evaluated | top-K from a candidate pool, 300 sampled users | **every unseen movie** ranked, **all** eligible users |
| Collaborative models | item-kNN only | item-kNN **and** SVD |
| Content features | TF-IDF over genre text | cosine on the 19 binary genre flags (already numeric, so TF-IDF adds nothing) |
| Reproducibility | notebook state | one command per stage, tests, CI |

## P1. The evaluation protocol: chronological leave-last-out

For every eligible user (≥ `MIN_USER_RATINGS` ratings) sort their ratings by time and cut the timeline into three:

```text
user's ratings over time  ─────────────────────────────────────────────►
[ ........ train ........ ] [ validation ] [ test ]
                              2nd-last       last
```

| Phase | Fit on | Score | Purpose |
|---|---|---|---|
| Validation | `train` | validation item | choose rating signals, tune hybrid weights |
| Test | `train + validation` | test item, **once** | the honest final number |

**Why this matters.** If you tune weights on the same item you report, your result is optimistic: you have fitted
the answer. Keeping the test item untouched until the very end is what makes the final table trustworthy.
It also respects time: the model never sees a rating that happened *after* the item it is asked to predict.

The demo below uses the same logic as `recommender/preprocessing.py` on a tiny table so you can see the cut.

In [ ]:
import pandas as pd

def leave_last_out(ratings):
    """Hold out each user's most recent rating. Returns (rest, held_out)."""
    ordered = ratings.sort_values(["user_id", "timestamp", "movie_id"]).reset_index(drop=True)
    last_rows = ordered.groupby("user_id").tail(1).index
    return ordered.drop(last_rows).reset_index(drop=True), ordered.loc[last_rows].reset_index(drop=True)

toy = pd.DataFrame({
    "user_id":   [1, 1, 1, 1, 1, 1,   2, 2, 2, 2, 2],
    "movie_id":  [10, 11, 12, 13, 14, 15,   10, 12, 16, 17, 18],
    "rating":    [5, 4, 3, 5, 4, 2,   4, 4, 5, 3, 5],
    "timestamp": [1, 2, 3, 4, 5, 6,   1, 2, 3, 4, 5],
})

train_full, test = leave_last_out(toy)      # last item  -> test
train, val = leave_last_out(train_full)     # next-last  -> validation

print("TRAIN\n", train[["user_id", "movie_id", "timestamp"]].to_string(index=False))
print("\nVALIDATION\n", val[["user_id", "movie_id", "timestamp"]].to_string(index=False))
print("\nTEST\n", test[["user_id", "movie_id", "timestamp"]].to_string(index=False))

TRAIN
  user_id  movie_id  timestamp
       1        10          1
       1        11          2
       1        12          3
       1        13          4
       2        10          1
       2        12          2
       2        16          3

VALIDATION
  user_id  movie_id  timestamp
       1        14          5
       2        17          4

TEST
  user_id  movie_id  timestamp
       1        15          6
       2        18          5


## P2. Metrics when each user has exactly one held-out movie

We score **every movie the user has not seen** and record the **rank** of the held-out movie. Everything else follows from that rank `r`:

| Metric | Formula for one user | Reading |
|---|---|---|
| Recall@K (= HitRate@K) | `1` if `r ≤ K` else `0` | did it reach the top K? |
| Precision@K | `Recall@K / K` | only one item can be relevant |
| MAP@K | `1/r` if `r ≤ K` else `0` | rewards higher placement |
| NDCG@K | `1/log2(r+1)` if `r ≤ K` else `0` | smoother discount for placement |
| MRR | `1/r` | average reciprocal rank |

Three more numbers describe the *lists*, not the hit: **Coverage@10** (share of the catalogue ever recommended),
**AvgPopularity@10** (popularity bias) and **Diversity@10** (how different the 10 items are from each other).

**Ties.** `rank_of_target` gives tied items the *average* position. Without this a model that scores every movie
identically could get rank 1 by luck of ordering. The demo shows it.

In [ ]:
import numpy as np

def rank_of_target(scores, seen, target_idx):
    """1-based rank of each user's target among UNSEEN items (ties share the average position)."""
    masked = np.where(seen, -np.inf, scores)
    rows = np.arange(len(target_idx))
    target = masked[rows, target_idx][:, None]
    greater = (masked > target).sum(axis=1)
    equal_others = (masked == target).sum(axis=1) - 1
    return 1.0 + greater + equal_others / 2.0

def ranking_metrics(ranks, ks=(3,)):
    ranks = np.asarray(ranks, dtype=float)
    out = {}
    for k in ks:
        hit = ranks <= k
        out[f"Recall@{k}"] = float(hit.mean())
        out[f"MAP@{k}"] = float(np.where(hit, 1.0 / ranks, 0.0).mean())
        out[f"NDCG@{k}"] = float(np.where(hit, 1.0 / np.log2(ranks + 1), 0.0).mean())
    out["MRR"] = float((1.0 / ranks).mean())
    return out

# 3 users x 6 movies. seen=True means the user already watched it, so it cannot be recommended.
scores = np.array([[0.9, 0.1, 0.8, 0.3, 0.2, 0.7],
                   [0.2, 0.9, 0.4, 0.8, 0.1, 0.3],
                   [0.5, 0.5, 0.5, 0.5, 0.5, 0.5]])   # user 3: a model that scores everything the same
seen = np.zeros_like(scores, dtype=bool)
seen[0, 0] = True          # user 0 already saw movie 0 (their top score) -> excluded
target = np.array([2, 5, 3])

ranks = rank_of_target(scores, seen, target)
print("ranks:", ranks)                       # user 0 -> 1, user 1 -> 4, user 2 -> 3.5 (six-way tie shares the average position)
print({k: round(v, 4) for k, v in ranking_metrics(ranks).items()})

ranks: [1.  4.  3.5]
{'Recall@3': 0.3333, 'MAP@3': 0.3333, 'NDCG@3': 0.3333, 'MRR': 0.5119}


## P3. `evaluate.py` — the experiment runner

**What it does, in order**

1. **Load + validate** MovieLens (`validate_data` fails loudly on bad ratings, duplicates, orphan movie IDs).
2. **Split** with `temporal_split` and build one shared `Index` (user/movie ID → matrix position).
3. **Validation phase** — score `random`, `popularity`, `content`, and `knn`/`svd` under three rating signals
   (`centered`, `binary`, `raw`). Pick the best signal for each by validation **NDCG@10**.
4. **Tune the hybrid** — try every weight combination (step 0.1) over `knn, svd, content, popularity`,
   keep the best validation NDCG@10, and save `reports/best_config.json`.
5. **Test phase, once** — refit on `train + validation`, score the test item, and report Precision/Recall/MAP/NDCG/MRR
   plus Coverage, AvgPopularity and Diversity.
6. **Ablation** — remove one component at a time (renormalising the remaining weights) to see what each contributes.
7. **Write** `validation_results.csv`, `test_results.csv`, `model_comparison.png`.

**Design decisions worth explaining in a presentation**

- *Per-user min-max normalisation* (`normalize_components`) puts kNN, SVD, content and popularity scores on a 0–1
  scale **using only unseen movies**, so a blend of unlike numbers is meaningful.
- *Already-seen movies get `-inf`* in `combine`: you never recommend what the user has watched.
- *The weight grid is exhaustive, not random*: with 4 components and step 0.1 there are only 286 combinations
  (see demo below), cheap enough to try them all.
- *Everything is a matrix operation* (`scores` is users × movies), so evaluating all users is fast.

Usage: `python evaluate.py` (optional `--step 0.05`, `--data-dir`, `--reports-dir`).

<details><summary><b>Full source: evaluate.py</b></summary>

```python
"""Offline evaluation: compare models, tune hybrid weights, write reports.

Protocol (chronological leave-last-out, one held-out movie per user):
  1. Validation phase: fit on `train`, score the validation item.
     Pick the best signal variant for kNN and SVD, then tune hybrid weights.
  2. Test phase: refit on `train + validation`, score the test item ONCE,
     using the settings chosen in step 1. Nothing is tuned on the test set.
The chosen settings are saved to reports/best_config.json, which train.py reads.

Usage:
    python evaluate.py
"""

from __future__ import annotations

import argparse
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd

from recommender.collaborative_model import ItemKNN, SVDModel
from recommender.config import (
    BEST_CONFIG_PATH,
    COMPONENTS,
    DATA_DIR,
    MIN_USER_RATINGS,
    REPORTS_DIR,
)
from recommender.content_model import ContentModel
from recommender.data import load_movielens, validate_data
from recommender.evaluation import (
    catalog_coverage,
    intra_list_diversity,
    mean_popularity,
    rank_of_target,
    ranking_metrics,
)
from recommender.preprocessing import Index, build_matrix, temporal_split
from recommender.ranking import (
    PopularityModel,
    RandomModel,
    combine,
    normalize_components,
    top_k_indices,
)

KNN_SIGNALS = ("centered", "binary", "raw")
SVD_SIGNALS = ("centered", "binary", "raw")
SELECT_METRIC = "NDCG@10"


def score_models(models: dict, history: pd.DataFrame, index: Index) -> dict:
    """Score every user against every item with each model."""
    cache: dict[str, object] = {}
    out = {}
    for name, model in models.items():
        if model.signal not in cache:
            cache[model.signal] = build_matrix(history, index, model.signal)
        out[name] = model.score(cache[model.signal])
    return out


def targets_for(held_out: pd.DataFrame, index: Index) -> np.ndarray:
    movie_ids = held_out.set_index("user_id").loc[index.user_ids, "movie_id"]
    return movie_ids.map(index.movie_to_idx).to_numpy(dtype=int)


def weight_grid(names, step: float = 0.1):
    """Every weight combination that sums to 1 in multiples of `step`."""
    n = round(1 / step)
    for combo in itertools.product(range(n + 1), repeat=len(names)):
        if sum(combo) == n:
            yield {name: c / n for name, c in zip(names, combo, strict=True)}


def fit_components(train, movies, index, knn_signal, svd_signal) -> dict:
    return {
        "knn": ItemKNN(knn_signal).fit(train, index),
        "svd": SVDModel(svd_signal).fit(train, index),
        "content": ContentModel().fit(movies, index),
        "popularity": PopularityModel().fit(train, index),
    }


def evaluate_scores(scores, seen, targets) -> dict:
    return ranking_metrics(rank_of_target(scores, seen, targets))


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=DATA_DIR)
    parser.add_argument("--reports-dir", type=Path, default=REPORTS_DIR)
    parser.add_argument("--config-out", type=Path, default=BEST_CONFIG_PATH)
    parser.add_argument("--step", type=float, default=0.1, help="weight grid step")
    args = parser.parse_args()
    args.reports_dir.mkdir(parents=True, exist_ok=True)
    args.config_out.parent.mkdir(parents=True, exist_ok=True)

    movies, ratings = load_movielens(args.data_dir)
    validate_data(movies, ratings)
    splits = temporal_split(ratings, MIN_USER_RATINGS)
    index = Index.build(splits.train_full["user_id"], movies["movie_id"])
    print(f"Users: {index.n_users} | Movies: {index.n_items}")

    # ------------------------------------------------ 1. validation phase
    print("\n[1/3] Validation: comparing model variants")
    seen_val = build_matrix(splits.train, index, "binary").toarray() > 0
    target_val = targets_for(splits.val, index)

    variants = {"random": RandomModel(), "popularity": PopularityModel().fit(splits.train, index)}
    variants["content"] = ContentModel().fit(movies, index)
    for s in KNN_SIGNALS:
        variants[f"knn_{s}"] = ItemKNN(s).fit(splits.train, index)
    for s in SVD_SIGNALS:
        variants[f"svd_{s}"] = SVDModel(s).fit(splits.train, index)

    val_scores = score_models(variants, splits.train, index)
    val_table = pd.DataFrame(
        {n: evaluate_scores(s, seen_val, target_val) for n, s in val_scores.items()}
    ).T
    print(val_table[["Recall@10", "MAP@10", "NDCG@10", "MRR"]].round(4))

    best_knn = max(KNN_SIGNALS, key=lambda s: val_table.loc[f"knn_{s}", SELECT_METRIC])
    best_svd = max(SVD_SIGNALS, key=lambda s: val_table.loc[f"svd_{s}", SELECT_METRIC])
    print(f"\nBest kNN signal: {best_knn} | best SVD signal: {best_svd}")

    # ---------------------------------------- 2. tune hybrid weights (val)
    print("\n[2/3] Tuning hybrid weights on the validation set")
    components = {
        "knn": val_scores[f"knn_{best_knn}"],
        "svd": val_scores[f"svd_{best_svd}"],
        "content": val_scores["content"],
        "popularity": val_scores["popularity"],
    }
    normalized = normalize_components(components, seen_val)
    best_weights, best_value = None, -1.0
    for weights in weight_grid(COMPONENTS, args.step):
        final = combine(normalized, weights, seen_val)
        value = ranking_metrics(rank_of_target(final, seen_val, target_val), ks=(10,))[
            SELECT_METRIC
        ]
        if value > best_value:
            best_weights, best_value = weights, value
    print(f"Best weights: {best_weights}  (validation {SELECT_METRIC} = {best_value:.4f})")

    config = {"knn_signal": best_knn, "svd_signal": best_svd, "weights": best_weights}
    args.config_out.write_text(json.dumps(config, indent=2))
    val_table.round(4).to_csv(args.reports_dir / "validation_results.csv")

    # ------------------------------------------------ 3. test phase (once)
    print("\n[3/3] Test: refit on train+validation, score the held-out test item")
    seen_test = build_matrix(splits.train_full, index, "binary").toarray() > 0
    target_test = targets_for(splits.test, index)
    parts = fit_components(splits.train_full, movies, index, best_knn, best_svd)
    models = {"random": RandomModel(), **parts}
    raw = score_models(models, splits.train_full, index)
    norm = normalize_components({n: raw[n] for n in COMPONENTS}, seen_test)

    finals = {
        "random": raw["random"],
        "popularity": raw["popularity"],
        "content": raw["content"],
        f"knn ({best_knn})": raw["knn"],
        f"svd ({best_svd})": raw["svd"],
        "HYBRID (tuned)": combine(norm, best_weights, seen_test),
    }
    for dropped in COMPONENTS:  # ablation: remove one component, renormalise
        rest = {n: w for n, w in best_weights.items() if n != dropped}
        total = sum(rest.values())
        if total > 0:
            rest = {n: w / total for n, w in rest.items()}
            finals[f"hybrid without {dropped}"] = combine(norm, rest, seen_test)

    rows = {}
    for name, scores in finals.items():
        metrics = evaluate_scores(scores, seen_test, target_test)
        top = top_k_indices(scores, 10)
        metrics["Coverage@10"] = catalog_coverage(top, index.n_items)
        metrics["AvgPopularity@10"] = mean_popularity(top, parts["popularity"].item_counts)
        metrics["Diversity@10"] = intra_list_diversity(top, parts["content"].similarity)
        rows[name] = metrics
    test_table = pd.DataFrame(rows).T.round(4)
    test_table.to_csv(args.reports_dir / "test_results.csv")
    show = ["Precision@10", "Recall@10", "MAP@10", "NDCG@10", "MRR", "Coverage@10", "Diversity@10"]
    print(test_table[show])

    plot_results(test_table, args.reports_dir / "model_comparison.png")
    print(f"\nSaved config -> {args.config_out}")
    print(
        f"Saved reports -> {args.reports_dir}/ "
        "(validation_results.csv, test_results.csv, model_comparison.png)"
    )


def plot_results(table: pd.DataFrame, path: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    main_rows = [r for r in table.index if not r.startswith("hybrid without")]
    ax = table.loc[main_rows, ["Recall@10", "MAP@10", "NDCG@10"]].plot(kind="bar", figsize=(9, 4.5))
    ax.set_title("Test-set comparison (leave-last-out, K=10)")
    ax.set_ylabel("score")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


if __name__ == "__main__":
    main()
```

</details>

In [ ]:
import itertools

def weight_grid(names, step=0.1):
    """Every weight combination that sums to 1 in multiples of `step`."""
    n = round(1 / step)
    for combo in itertools.product(range(n + 1), repeat=len(names)):
        if sum(combo) == n:
            yield {name: c / n for name, c in zip(names, combo, strict=True)}

names = ("knn", "svd", "content", "popularity")
grid = list(weight_grid(names, 0.1))
print("combinations:", len(grid))                          # C(10+3, 3) = 286
print("all sum to 1:", all(abs(sum(w.values()) - 1) < 1e-9 for w in grid))
print("example:", grid[len(grid) // 2])
print("with step 0.05:", len(list(weight_grid(names, 0.05))), "combinations")

combinations: 286
all sum to 1: True
example: {'knn': 0.2, 'svd': 0.2, 'content': 0.5, 'popularity': 0.1}
with step 0.05: 1771 combinations


## P4. `train.py` — fit on all data, save one artifact

Evaluation answers *"which settings are best?"* using hold-outs. Production wants the **best possible model**,
so `train.py` uses **every rating**, applies the settings from `best_config.json` (or the defaults in `config.py`
if you never ran `evaluate.py`), and saves a single file:

```text
python train.py  →  artifacts/recommender.joblib
```

The saved object is the whole `Recommender` (index, titles, four fitted models, history matrices). The API only
needs this one file; it does **not** need the data, the notebook, or the training code path.

> **Security note.** `joblib` files are pickles. Never load one from someone you do not trust, and load with the
> same library versions you trained with. This is also why `artifacts/` is in `.gitignore`.

<details><summary><b>Full source: train.py</b></summary>

```python
"""Train the final model on ALL ratings and save it as an artifact.

Usage:
    python train.py
Run `python evaluate.py` first if you want tuned weights (best_config.json).
"""

from __future__ import annotations

import argparse
from pathlib import Path

from recommender.config import ARTIFACT_PATH, BEST_CONFIG_PATH, DATA_DIR
from recommender.data import load_movielens
from recommender.inference import Recommender, load_best_config


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=DATA_DIR)
    parser.add_argument("--out", type=Path, default=ARTIFACT_PATH)
    parser.add_argument("--config", type=Path, default=BEST_CONFIG_PATH)
    args = parser.parse_args()

    movies, ratings = load_movielens(args.data_dir)
    config = load_best_config(args.config)
    print(f"Config: {config}")

    model = Recommender(
        knn_signal=config["knn_signal"],
        svd_signal=config["svd_signal"],
        weights=config["weights"],
    ).fit(movies, ratings)
    model.save(args.out)

    print(f"Saved {args.out.resolve()}")
    print(
        f"Movies: {len(movies):,} | Ratings: {len(ratings):,} | "
        f"Users: {ratings['user_id'].nunique():,}"
    )
    sample_user = int(model.index.user_ids[0])
    print(f"\nSample recommendations for user {sample_user}:")
    print(model.recommend_for_user(sample_user, k=10)[["movie_id", "title", "score"]])


if __name__ == "__main__":
    main()
```

</details>

## P5. `api.py` — the inference service

**Rule: the API serves, it never trains.** It loads the artifact once at startup and answers requests from memory.

| Endpoint | Purpose | Notable behaviour |
|---|---|---|
| `GET /health` | liveness + model status | reports `model_loaded` |
| `GET /recommendations/user/{user_id}` | hybrid ranking | unknown user → **cold start** (strategy says so) |
| `GET /recommendations/movie/{movie_id}` | similar movies | `mode=content` or `collaborative`; unknown movie → **404** |
| `GET /recommendations/popular` | popularity ranking | optional `preferred_genres` boost |

**Patterns to point out**

- **Application factory** `create_app(model=None, model_path=None)`: tests can inject a tiny model directly
  without touching disk.
- **Lifespan** loads the artifact on startup (path from `MODEL_PATH`, falling back to `config.ARTIFACT_PATH`).
- **Graceful failure**: no artifact → `/health` still answers and recommendation routes return **503** with a
  helpful message instead of crashing.
- **Validation at the edge**: `k` is bounded by `Query(DEFAULT_K, ge=1, le=MAX_K)`; bad genre names become **400**;
  unknown movie becomes **404**.
- **Typed responses** (`RecommendationResponse`) give automatic documentation at `/docs` and a stable contract.
- **Transparency**: personalised results include per-component scores (`components`), so you can explain *why* a
  movie was recommended.

Try it:

```bash
uvicorn api:app --reload
curl "http://127.0.0.1:8000/recommendations/user/1?k=5"
curl "http://127.0.0.1:8000/recommendations/user/999999?preferred_genres=Comedy"   # cold start
curl "http://127.0.0.1:8000/recommendations/movie/50?mode=collaborative"
```

<details><summary><b>Full source: api.py</b></summary>

```python
"""FastAPI inference layer. Serves a pre-trained artifact; never trains."""

from __future__ import annotations

import os
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Literal

from fastapi import FastAPI, HTTPException, Query, Request
from pydantic import BaseModel

from recommender.config import ARTIFACT_PATH, DEFAULT_K, MAX_K
from recommender.inference import Recommender, UnknownMovieError


class Recommendation(BaseModel):
    movie_id: int
    title: str
    genres: list[str]
    score: float
    components: dict[str, float] | None = None


class RecommendationResponse(BaseModel):
    strategy: Literal["hybrid", "cold_start", "similar_content", "similar_collaborative", "popular"]
    items: list[Recommendation]


def _records(frame) -> list[Recommendation]:
    return [Recommendation(**row) for row in frame.to_dict(orient="records")]


def create_app(model: Recommender | None = None, model_path: str | Path | None = None) -> FastAPI:
    """Application factory. Pass `model` directly in tests."""
    path = Path(model_path or os.environ.get("MODEL_PATH", ARTIFACT_PATH))

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        app.state.model = model
        if app.state.model is None and path.exists():
            app.state.model = Recommender.load(path)
        yield

    app = FastAPI(title="Movie Recommendation API", version="2.0.0", lifespan=lifespan)

    def get_model(request: Request) -> Recommender:
        loaded = request.app.state.model
        if loaded is None:
            raise HTTPException(503, "Model artifact not loaded. Run `python train.py` first.")
        return loaded

    @app.get("/health")
    def health(request: Request):
        return {"status": "ok", "model_loaded": request.app.state.model is not None}

    @app.get("/recommendations/popular", response_model=RecommendationResponse)
    def popular(
        request: Request,
        k: int = Query(DEFAULT_K, ge=1, le=MAX_K),
        preferred_genres: list[str] | None = Query(None),
    ):
        rec = get_model(request)
        try:
            frame = rec.popular(k, preferred_genres)
        except ValueError as exc:
            raise HTTPException(400, str(exc)) from exc
        return RecommendationResponse(strategy="popular", items=_records(frame))

    @app.get("/recommendations/user/{user_id}", response_model=RecommendationResponse)
    def for_user(
        request: Request,
        user_id: int,
        k: int = Query(DEFAULT_K, ge=1, le=MAX_K),
        preferred_genres: list[str] | None = Query(None),
    ):
        rec = get_model(request)
        try:
            frame = rec.recommend_for_user(user_id, k, preferred_genres)
        except ValueError as exc:
            raise HTTPException(400, str(exc)) from exc
        strategy = "hybrid" if rec.has_user(user_id) else "cold_start"
        return RecommendationResponse(strategy=strategy, items=_records(frame))

    @app.get("/recommendations/movie/{movie_id}", response_model=RecommendationResponse)
    def for_movie(
        request: Request,
        movie_id: int,
        k: int = Query(DEFAULT_K, ge=1, le=MAX_K),
        mode: Literal["content", "collaborative"] = "content",
    ):
        rec = get_model(request)
        try:
            frame = rec.similar_movies(movie_id, k, mode)
        except UnknownMovieError:
            raise HTTPException(404, f"Unknown movie_id {movie_id}") from None
        return RecommendationResponse(strategy=f"similar_{mode}", items=_records(frame))

    return app


app = create_app()
```

</details>

## P6. Docker — same behaviour on every machine

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY recommender/ recommender/
COPY api.py .
# The trained artifact is created by `python train.py` BEFORE building the image.
COPY artifacts/recommender.joblib artifacts/recommender.joblib

ENV MODEL_PATH=artifacts/recommender.joblib
EXPOSE 8000
CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Why it is written this way**

- `requirements.txt` is copied and installed **before** the code, so Docker reuses the cached dependency layer
  when only your code changes (fast rebuilds).
- Only `recommender/`, `api.py` and the **trained artifact** are copied in; data, tests, reports and notebooks stay out
  (see `.dockerignore`), keeping the image small.
- The artifact is built *outside* the image (`python train.py`) and copied in, which keeps training out of the
  serving container, matching Part K's separation.
- `MODEL_PATH` is an environment variable, so the same image can serve a different artifact without a rebuild.

```bash
python train.py
docker build -t movie-recommender .
docker run -p 8000:8000 movie-recommender
# open http://localhost:8000/docs
```

**`.gitignore` vs `.dockerignore`:** `data/` and `artifacts/` are git-ignored (large, regenerable), so anyone cloning
the repo must run `python train.py` before `docker build`.

<details><summary><b>.dockerignore and .gitignore</b></summary>

```text
.git
.github
.venv
__pycache__
*.pyc
data
notebooks
tests
reports
.pytest_cache
.ruff_cache
```

```text
# Python
__pycache__/
*.pyc
.venv/
venv/
.env
.pytest_cache/
.ruff_cache/
.ipynb_checkpoints/

# Data and generated model files (never commit these)
data/
artifacts/

# OS / editor
.DS_Store
.vscode/
.idea/
```

</details>

## P7. Tests, linting and CI

| Tool | Config | What it protects |
|---|---|---|
| `pytest` (42 tests) | `pytest.ini` | metrics, splitting, matrix building, models, ranking, recommender, API |
| `ruff` | `ruff.toml` (rules E, F, I, B; line length 100) | style, unused imports, common bugs |
| GitHub Actions | `.github/workflows/ci.yml` | runs `ruff check .` and `pytest -q` on every push to `main` and every pull request |

What the tests verify (examples): ties are ranked fairly, seen items never get recommended, unknown users fall back to
cold start, unknown movies return 404, `k` is clipped, and invalid genres are rejected. A synthetic dataset
(`tests/synthetic.py`) keeps tests fast and independent of the real download.

```bash
ruff check .
pytest -q
```

On the project as uploaded, `ruff check .` passes and all 42 tests pass.

<details><summary><b>pytest.ini, ruff.toml, requirements.txt</b></summary>

```ini
[pytest]
testpaths = tests
pythonpath = .
```

```toml
line-length = 100
extend-exclude = ["*.ipynb"]

[lint]
select = ["E", "F", "I", "B"]

[lint.flake8-bugbear]
extend-immutable-calls = ["fastapi.Query"]
```

```text
pandas>=2.0
numpy>=1.24
scipy>=1.10
scikit-learn>=1.3
joblib>=1.3
matplotlib>=3.7
fastapi>=0.110
uvicorn[standard]>=0.27
httpx>=0.27
pytest>=8.0
```

</details>

## P8. Results and how to read them honestly

From `python evaluate.py` (test set: one held-out item per eligible user; full numbers in `reports/test_results.csv`):

| Model | Recall@10 (= HitRate) | MAP@10 | NDCG@10 | MRR | Coverage@10 |
|---|---|---|---|---|---|
| Random | 0.0053 | 0.0015 | 0.0024 | 0.0046 | 0.9958 |
| Popularity | 0.0297 | 0.0082 | 0.0130 | 0.0134 | 0.0059 |
| Content (genres) | 0.0201 | 0.0064 | 0.0097 | 0.0104 | 0.4566 |
| Item-kNN | 0.0827 | 0.0239 | 0.0375 | 0.0364 | 0.1950 |
| SVD | 0.0827 | 0.0240 | 0.0375 | 0.0372 | 0.1350 |
| **Hybrid (tuned)** | **0.0838** | **0.0245** | **0.0382** | **0.0378** | 0.2812 |

Tuned weights: **SVD 0.9, content 0.1, kNN 0, popularity 0** (both rating signals chosen: `raw`).

**What the numbers say**

- Collaborative models beat popularity by about **3×** on Recall@10 and beat random by about **15×**.
- **The hybrid's edge over SVD is tiny** (0.0838 vs 0.0827). With one test item per user this is almost certainly
  noise. Say so in your presentation: it shows you understand the evidence.
- **Ablation:** removing SVD collapses the hybrid to content-only performance (Recall@10 0.0201), so SVD carries the
  model. Removing kNN or popularity changes nothing, because their tuned weight is already 0.
- **Random has ~100% coverage, popularity ~0.6%.** High coverage alone is not quality; random is "diverse" and useless.
  The useful comparison is accuracy *and* coverage together.
- **Content-only diversity is very low** (≈0.09): many movies share identical genre vectors, so its top-10 lists are
  near-duplicates. That is the strongest argument for richer content features.

**Limitations (be upfront)**: MovieLens 100K is a benchmark, not live traffic; one test item per user is noisy;
genre-only content features are coarse; dense similarity matrices suit ~1.7k movies and would need approximate
nearest neighbours at scale; brand-new movies only get content-based scores; the artifact is a pickle.

## P9. Version control: how a project is meant to be built

Graders and employers read a commit history as evidence of **how** you worked. Healthy history looks like many small,
focused commits that tell a story, for example:

```text
Add data loading and validation
Add chronological leave-last-out split with tests
Add ItemKNN and SVD models
Tune hybrid weights on validation set
Add FastAPI inference layer with 503/404 handling
Add Dockerfile and .dockerignore
Add CI workflow (ruff + pytest)
Document results and limitations in README
```

Good habits: commit after each working step, write the message as *"what this change does"*, keep generated files
(`data/`, `artifacts/`) out of git, and run `ruff check .` and `pytest -q` before pushing. Check your history any time with:

```bash
git log --oneline --date=short --pretty=format:"%h %ad %s"
```

## P10. Where to take it next

| Idea | Why it is worth doing |
|---|---|
| Add richer content features (title text, release year, TMDB metadata) | fixes the low content diversity seen above |
| MMR / diversity re-ranking | directly improves Diversity@10 without retraining |
| Implicit-feedback models (ALS / BPR) | the standard next step beyond PureSVD |
| Confidence intervals / multiple seeds in `evaluate.py` | turns "probably noise" into a measured statement |
| Model versioning + `/health` showing the version | safe rollbacks and traceable results |
| A small web front-end on top of the API | the most convincing thing to show in a demo |

# Final takeaway

The progression from this course should now look like:

```text
Python
  ↓
Data Analysis
  ↓
Supervised Learning
  ↓
Unsupervised Learning
  ↓
Clustering
  ↓
Similarity
  ↓
Recommendation Systems
  ↓
Model Evaluation
  ↓
Feature Engineering
  ↓
Hybrid Ranking
  ↓
Model Persistence
  ↓
API
  ↓
Docker / Cloud
  ↓
Monitoring
  ↓
Production ML System
```

The most important lesson is:

> **A production ML system is much bigger than the model.**

The model is one component inside a complete data, engineering, evaluation and deployment pipeline.